In [10]:
# ============================================================
# Free Token Usage Tracker for Google Gemini (Colab / Notebook)
# ============================================================

# ---------- 1. Install Dependencies ----------
# Uncomment and run the line below if using a fresh Colab session
# !pip install google-genai pandas plotly sqlalchemy

import os
import pandas as pd
import plotly.express as px
from google import genai
from google.colab import userdata
from sqlalchemy import create_engine, Column, Integer, String, Float, DateTime, text
from sqlalchemy.orm import declarative_base, sessionmaker
from datetime import datetime, timezone, timedelta
from IPython.display import display

# ---------- 2. Database Setup ----------
Base = declarative_base()

class TokenUsage(Base):
    __tablename__ = 'token_usage'
    id = Column(Integer, primary_key=True)
    timestamp = Column(DateTime, default=lambda: datetime.now(timezone.utc))
    model = Column(String)
    prompt_tokens = Column(Integer)
    completion_tokens = Column(Integer)
    total_tokens = Column(Integer)
    cost = Column(Float, default=0.0)

# Creates a local database file in your Colab environment
engine = create_engine('sqlite:///usage.db')
Base.metadata.create_all(engine)
Session = sessionmaker(bind=engine)

# ---------- 3. Gemini Tracker Wrapper ----------
class TrackedGemini:
    def __init__(self, api_key=None):
        # Initializes using the official google-genai SDK
        self.client = genai.Client(api_key=api_key)
        # 🐛 FIXED: Updated to the current active free-tier model
        self.default_model = "gemini-3.5-flash"

    def generate(self, prompt, model=None, stream=False, **kwargs):
        model_name = model or self.default_model

        if stream:
            response = self.client.models.generate_content_stream(
                model=model_name,
                contents=prompt,
                **kwargs
            )
            return self._handle_stream(response, model_name)
        else:
            response = self.client.models.generate_content(
                model=model_name,
                contents=prompt,
                **kwargs
            )
            self._log_usage(response, model_name)
            return response

    def _handle_stream(self, stream, model_name):
        usage = None
        try:
            for chunk in stream:
                if hasattr(chunk, 'usage_metadata') and chunk.usage_metadata:
                    usage = chunk.usage_metadata
                yield chunk
        finally:
            if usage:
                self._log_usage_from_metadata(usage, model_name)

    def _log_usage(self, response, model_name):
        try:
            if hasattr(response, 'usage_metadata') and response.usage_metadata:
                self._log_usage_from_metadata(response.usage_metadata, model_name)
        except Exception as e:
            print(f"Could not extract usage: {e}")

    def _log_usage_from_metadata(self, metadata, model_name):
        prompt_tokens = getattr(metadata, 'prompt_token_count', 0)
        completion_tokens = getattr(metadata, 'candidates_token_count', 0)
        total_tokens = getattr(metadata, 'total_token_count', prompt_tokens + completion_tokens)

        session = Session()
        try:
            record = TokenUsage(
                timestamp=datetime.now(timezone.utc),
                model=model_name,
                prompt_tokens=prompt_tokens,
                completion_tokens=completion_tokens,
                total_tokens=total_tokens,
                cost=0.0
            )
            session.add(record)
            session.commit()
        except Exception as e:
            session.rollback()
            print(f"Error logging usage: {e}")
        finally:
            session.close()

# ---------- 4. Reporting Function ----------
def show_report(days=7):
    cutoff = datetime.now(timezone.utc) - timedelta(days=days)

    query = text("SELECT * FROM token_usage WHERE timestamp >= :cutoff ORDER BY timestamp DESC")

    with engine.connect() as conn:
        df = pd.read_sql(query, conn, params={"cutoff": cutoff})

    if df.empty:
        print("No usage data recorded yet. Run a prompt first.")
        return

    df['timestamp'] = pd.to_datetime(df['timestamp'])

    # KPIs
    total_cost = df['cost'].sum()
    total_tokens = df['total_tokens'].sum()
    total_requests = len(df)

    print(f"\n📊 Last {days} days summary:")
    print(f"   Requests: {total_requests}")
    print(f"   Total tokens: {total_tokens:,}")
    print(f"   Total cost (free tier): ${total_cost:.2f}\n")

    # Daily Token Plot
    daily = df.set_index('timestamp').resample('D')['total_tokens'].sum().reset_index()
    if not daily.empty:
        fig1 = px.line(daily, x='timestamp', y='total_tokens', title='Daily Token Usage')
        fig1.show()

    # Breakdown table
    token_agg = df.groupby('model')[['prompt_tokens', 'completion_tokens', 'total_tokens']].sum()
    print("Token breakdown by model:")
    display(token_agg)

print("✅ Tracker setup complete!")


# ============================================================
# 5. TEST RUN USING COLAB SECRETS
# ============================================================
try:
    # Fetches your secret named GEMINI_API_KEY from Colab Secrets (🔑)
    api_key = userdata.get('GEMINI_API_KEY')
    client = TrackedGemini(api_key=api_key)

    print("\nSending test prompt...")
    response = client.generate("Explain artificial intelligence in one short sentence.")

    print("\nResponse:")
    print(response.text)

    print("\nGenerating Usage Report...")
    show_report()

except Exception as e:
    print(f"❌ Error: {e}")
    print("Make sure you added 'GEMINI_API_KEY' under Colab Secrets (🔑 icon) and enabled 'Notebook access'.")


✅ Tracker setup complete!

Sending test prompt...

Response:
Artificial intelligence is technology that enables computers to think, learn, and solve problems like humans.

Generating Usage Report...

📊 Last 7 days summary:
   Requests: 1
   Total tokens: 557
   Total cost (free tier): $0.00



Token breakdown by model:


,prompt_tokens,completion_tokens,total_tokens
model,,,
gemini-3.5-flash,9,18,557
